# Data transformation to feed the STKDE method / Transformación de datos para alimentar el método STKDE

**[EN]**

This document outlines the process of preparing the unified gender-based crime dataset (`unified_selected_crime_dataset.parquet`) to feed the spatiotemporal density estimator (STKDE) of the web prototype.

**Scope:** Only the columns necessary for density estimation are retained. The web system's analytical dashboards will consume a different dataset; therefore, descriptive attributes (type of crime, borough, neighborhood, etc.) are not preserved.

**Naming Convention:** The original names of the dataset are retained (`COORD. X`, `COORD. Y`, `FECHA DE LOS HECHOS`, `HORA DE LOS HECHOS`).

**[ES]**

Este cuaderno documenta el proceso de preparación del dataset unificado de delitos de género
(`unified_selected_crime_dataset.parquet`) para alimentar el estimador de densidad
espacio-temporal (STKDE) del prototipo web.

**Alcance:** solo mantener las columnas necesarias para la estimación de densidad. Los dashboards
analíticos del sistema web consumirán otro dataset; por ello no se conservan atributos
descriptivos (tipo de delito, alcaldía, colonia, etc.).

**Convención de nombres:** se mantienen los nombres originales del parquet
(`COORD. X`, `COORD. Y`, `FECHA DE LOS HECHOS`, `HORA DE LOS HECHOS`).

In [ ]:
# Import libraries / Importar librerías

import pandas as pd

In [ ]:
STKDE_COLUMNS = [
    "COORD. X",
    "COORD. Y",
    "FECHA DE LOS HECHOS",
    "HORA DE LOS HECHOS",
]

## <font color='#1083D6'>Loading the unified dataset / Carga del dataset unificado </font>

**[EN]**

The file contains approximately 41,935 records (2020–2025) with administrative and geospatial columns. The coordinates are already in WGS84 (decimal degrees).

**[ES]**

El archivo contiene 41 935 registros (2020–2025) con columnas administrativas y
geoespaciales. Las coordenadas ya están en WGS84 (grados decimales).

In [ ]:
df_raw = pd.read_parquet('../data/inter/unified_selected_crime_dataset.parquet')

print(f"Filas: {len(df_raw):,}  |  Columnas: {df_raw.shape[1]}")
print("\nEsquema original:")
print(df_raw.dtypes.to_string())
df_raw.head(3)

## <font color='#1083D6'>Removing unnecessary columns for STKDE / Eliminación de columnas innecesarias para STKDE</font>

**[EN]**

To estimate the density at a point `(x, y, t)`, only the following are required:

| Column | Role in STKDE |
|---|---|
| **X COORD.** | Longitude (spatial axis) |
| **Y COORD.** | Latitude (spatial axis) |
| **DATE OF INCIDENT** | Date component of the incident time |
| **TIME OF INCIDENT** | Time component of the incident time |

The rest (IDs, streets, neighborhood, borough, type of crime, observations, etc.) are discarded
because they are not part of the spatiotemporal kernel and because the dashboards will use a different source.

**[ES]**

Para estimar la densidad en un punto `(x, y, t)` solo se requieren:

| Columna | Rol en STKDE |
|---|---|
| **COORD. X** | Longitud (eje espacial) |
| **COORD. Y** | Latitud (eje espacial) |
| **FECHA DE LOS HECHOS** | Componente de fecha del instante del incidente |
| **HORA DE LOS HECHOS** | Componente horaria del instante del incidente |

El resto (IDs, calles, colonia, alcaldía, tipo de delito, observaciones, etc.) se descarta
porque no entra en el kernel espacio-temporal y porque los dashboards usarán otra fuente.

In [ ]:
df = df_raw[STKDE_COLUMNS].copy()

dropped = sorted(set(df_raw.columns) - set(STKDE_COLUMNS))
print(f"Columnas conservadas ({len(STKDE_COLUMNS)}): {STKDE_COLUMNS}")
print(f"Columnas eliminadas ({len(dropped)}): {dropped}")
print(f"\nShape tras selección: {df.shape}")
df.head(3)

## <font color='#1083D6'>Data type conversion for STKDE method feed / Conversión de tipos de dato para la alimentación del método STKDE</font>

**[EN]**

The project's STKDE estimator expects:

| Column | Source Type (parquet) | Destination Type | Format |
| ---|---|---|---|
| **X COORD.** | `float64` | `float64` | WGS84 Longitude (≈ −99.xx) |
| **Y COORD.** | `float64` | `float64` | WGS84 Latitude (≈ 19.xx) |
| **DATE OF EVENTS** | `datetime64[us]` | `str` | YYYY-MM-DD |
| **TIME OF EVENTS** | `datetime64[us]` | `int64` | Integer 0–23 |

The complete instant will be reconstructed as:
`DATE OF EVENTS + " " + TIME OF EVENTS + ":00:00"`.

**[ES]**

El estimador STKDE del proyecto espera:

| Columna | Tipo origen (parquet) | Tipo destino | Formato |
|---|---|---|---|
| **COORD. X** | `float64` | `float64` | Longitud WGS84 (≈ −99.xx) |
| **COORD. Y** | `float64` | `float64` | Latitud WGS84 (≈ 19.xx) |
| **FECHA DE LOS HECHOS** | `datetime64[us]` | `str` | YYYY-MM-DD |
| **HORA DE LOS HECHOS** | `datetime64[us]` | `int64` | Entero 0–23 |

El instante completo se reconstruirá como:
`FECHA DE LOS HECHOS + " " + HORA DE LOS HECHOS + ":00:00"`.

In [ ]:
print("Tipos ANTES de la conversión:")
print(df.dtypes.to_string())
print("\nMuestra HORA DE LOS HECHOS (datetime):")
print(df["HORA DE LOS HECHOS"].head(5).tolist())

In [ ]:
# Coordenadas: asegurar float64 (ya vienen en WGS84; no requieren reproyección)
df["COORD. X"] = pd.to_numeric(df["COORD. X"], errors="coerce").astype("float64")
df["COORD. Y"] = pd.to_numeric(df["COORD. Y"], errors="coerce").astype("float64")

# Fecha: datetime64 → string YYYY-MM-DD
df["FECHA DE LOS HECHOS"] = (
    pd.to_datetime(df["FECHA DE LOS HECHOS"], errors="coerce")
    .dt.strftime("%Y-%m-%d")
    .astype(str)
)

# Hora: datetime64 (1900-01-01 HH:MM:SS) → entero 0–23
df["HORA DE LOS HECHOS"] = (
    pd.to_datetime(df["HORA DE LOS HECHOS"], errors="coerce")
    .dt.hour
    .astype("Int64")
)

print("Tipos DESPUÉS de la conversión:")
print(df.dtypes.to_string())
df.head(5)